# 02 — Data Cleaning

Purpose: clean types, validate the IAM risk catalogue, safely merge risk metadata, and save a clean dataset.
Missing values are not blindly filled here; the ML preprocessing pipelines handle numeric/categorical imputation inside each training fold.

In [1]:
# AI-Based IAM Permission Optimizer — Model V2
# Run notebooks in order: 01 → 10
# Raw CSVs should be available in the project root or adjust RAW_DIR below.
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

usage = pd.read_csv(RAW_DIR / "ml_ready_synthetic.csv")
risk = pd.read_csv(RAW_DIR / "AWS_Risk_Weight_Table_FINAL_4007_Actions_Shuffled.csv")

usage.columns = usage.columns.str.strip()
risk.columns = risk.columns.str.strip()

print("Loaded:", usage.shape, risk.shape)

Loaded: (20000, 25) (4005, 6)


In [2]:
# Standardize string fields
usage_string_cols = ["user_id", "role_id", "action", "resource", "resource_scope", "permission_status"]
for col in usage_string_cols:
    if col in usage.columns:
        usage[col] = usage[col].astype("string").str.strip()

for col in ["action", "service", "operation_type", "risk_level"]:
    if col in risk.columns:
        risk[col] = risk[col].astype("string").str.strip()

# Numeric fields
numeric_cols = [
    "usage_count", "unique_days_used", "days_since_last_use",
    "success_count", "failure_count", "success_rate", "failure_rate", "risk_weight"
]
for col in numeric_cols:
    if col in usage.columns:
        usage[col] = pd.to_numeric(usage[col], errors="coerce")

if "risk_weight" in risk.columns:
    risk["risk_weight"] = pd.to_numeric(risk["risk_weight"], errors="coerce")

for col in ["first_used", "last_used"]:
    if col in usage.columns:
        usage[col] = pd.to_datetime(usage[col], errors="coerce", utc=True)

print("Type normalization complete.")

Type normalization complete.


In [3]:
# Validate risk catalogue is one row per action with no conflicting metadata.
metadata_cols = ["service", "operation_type", "risk_level", "risk_weight"]
conflict_mask = (
    risk.groupby("action")[metadata_cols]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
)

if conflict_mask.any():
    bad_actions = conflict_mask[conflict_mask].index.tolist()
    raise ValueError(f"Conflicting risk metadata for actions: {bad_actions[:20]}")

risk_catalog = (
    risk.sort_values("action")
    .drop_duplicates("action")
    [["action", "service", "operation_type", "risk_level", "risk_weight", *( ["reason"] if "reason" in risk.columns else [] )]]
)

print("Unique risk actions:", risk_catalog["action"].nunique())

Unique risk actions: 4005


In [4]:
# Safe many-to-one merge
df = usage.merge(
    risk_catalog,
    on="action",
    how="left",
    validate="many_to_one",
    suffixes=("", "_catalog")
)

missing_catalog = df["service"].isna().sum()
print("Rows without a risk catalogue match:", int(missing_catalog))

if missing_catalog:
    actions = df.loc[df["service"].isna(), "action"].drop_duplicates().tolist()
    raise ValueError(f"Unmatched actions: {actions[:20]}")

# Compare existing metadata with catalog metadata when both exist.
checks = [
    ("operation_type", "operation_type_catalog"),
    ("risk_level", "risk_level_catalog"),
    ("risk_weight", "risk_weight_catalog")
]
for source, catalog_col in checks:
    if catalog_col in df.columns and source in usage.columns:
        if source == "risk_weight":
            mismatch = ~np.isclose(df[source], df[catalog_col], equal_nan=True)
        else:
            mismatch = df[source].astype("string") != df[catalog_col].astype("string")
        print(f"{source} mismatches:", int(mismatch.sum()))
        if mismatch.any():
            raise ValueError(f"Risk metadata mismatch detected in {source}")

# Always use the validated catalogue as the source of truth.
for col in ["operation_type", "risk_level", "risk_weight"]:
    catalog_col = f"{col}_catalog"
    if catalog_col in df.columns:
        df[col] = df[catalog_col]

# Drop merge-helper columns
helper_cols = [c for c in df.columns if c.endswith("_catalog") and c != "catalog_reason"]
df = df.drop(columns=helper_cols, errors="ignore")

print("Cleaned shape:", df.shape)

Rows without a risk catalogue match: 0
operation_type mismatches: 0
risk_level mismatches: 0
risk_weight mismatches: 0
Cleaned shape: (20000, 26)


In [5]:
# Reject impossible values rather than silently changing them.
checks = {
    "usage_count": df["usage_count"].notna() & (df["usage_count"] < 0),
    "unique_days_used": df["unique_days_used"].notna() & (df["unique_days_used"] < 0),
    "days_since_last_use": df["days_since_last_use"].notna() & (df["days_since_last_use"] < 0),
    "success_count": df["success_count"].notna() & (df["success_count"] < 0),
    "failure_count": df["failure_count"].notna() & (df["failure_count"] < 0),
}

for name, mask in checks.items():
    print(f"Invalid {name} rows:", int(mask.sum()))
    if mask.any():
        raise ValueError(f"Negative values found in {name}")

count_mismatch = (
    df["success_count"].fillna(0) + df["failure_count"].fillna(0)
    != df["usage_count"].fillna(0)
)
print("Count-consistency mismatches:", int(count_mismatch.sum()))

if count_mismatch.any():
    raise ValueError("success_count + failure_count does not equal usage_count for some rows.")

Invalid usage_count rows: 0
Invalid unique_days_used rows: 0
Invalid days_since_last_use rows: 0
Invalid success_count rows: 0
Invalid failure_count rows: 0
Count-consistency mismatches: 0


In [6]:
# Stable row identifier used by all later notebooks.
if df["id"].duplicated().any():
    raise ValueError("The input id column is not unique. Fix IDs before creating persistent splits.")

df["row_id"] = df["id"].astype(str)

clean_path = DATA_DIR / "cleaned_dataset.csv"
df.to_csv(clean_path, index=False)

print("Saved:", clean_path)

Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\data\cleaned_dataset.csv
